# Fontes, população e seleção

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_02/01_fontes_populacao_e_selecao.ipynb)

O guia definiu o protocolo como produto da unidade. Este notebook inicia
sua construção pela decisão que condiciona todas as demais: quais casos
podem integrar a base e com base em quais evidências.

## 1. A base começa antes da planilha

Retome sua pergunta da Unidade 1. A **população de interesse** reúne os
casos sobre os quais se deseja argumentar; a **população acessível** reúne
os casos que podem ser localizados e consultados nas condições do projeto;
o **corpus** é o conjunto efetivamente delimitado por critérios explícitos.

Essas extensões não são automaticamente iguais. Um acervo digital costuma
refletir preservação, catalogação, digitalização, acesso e decisões
institucionais anteriores à pesquisa.

Observe como o diagrama situa o corpus dentro da população acessível e
explicita as mediações que limitam a passagem entre os conjuntos.

![População de interesse contém a população acessível, que contém o corpus; ao lado, produção, preservação, catalogação, localização, digitalização, acesso e seleção aparecem como mediações.](imagens/01_populacao_acessivel_corpus.svg)

A figura é um modelo conceitual, não uma prova de representatividade. Um
corpus pode ser grande e ainda assim resultar de acesso muito desigual.

Delimitados os três conjuntos, é preciso examinar o papel de cada material
na argumentação. Essa relação com a pergunta determina se ele funciona
como fonte primária, secundária ou dado derivado.

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_02'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = []
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

## 2. Fonte primária, secundária e dado derivado

Na prática historiográfica, a distinção pode ser formulada assim:

| Relação com a pesquisa | Definição operacional |
|---|---|
| **Fonte primária** | Documento, objeto, imagem, registro estatístico, testemunho ou outro vestígio produzido no período estudado ou por atores diretamente relacionados a ele, mobilizado como evidência sobre esse contexto. |
| **Fonte secundária** | Interpretação posterior que analisa o passado com base em fontes primárias e em diálogo com outras interpretações. Livros e artigos historiográficos são exemplos recorrentes. |

A American Historical Association (AHA, 2023) emprega uma noção ampla
de documento primário — que inclui textos, artefatos, imagens, vídeos,
estatísticas, relatos orais e ambientes construídos — e define a
literatura secundária como interpretações posteriores fundamentadas
nesses documentos. A própria associação adverte, contudo, que a fronteira
entre as duas categorias depende em grande medida da pergunta de pesquisa.

O mesmo catálogo ocupa posições diferentes no diagrama conforme o que se
pretende investigar.

![O mesmo catálogo institucional funciona como instrumento ou dado derivado para uma pergunta sobre cartas e como fonte primária para uma pergunta sobre práticas de catalogação.](imagens/01_papel_das_fontes.svg)

Portanto, “primária” e “secundária” descrevem uma **relação entre o
material, o problema e o uso analítico**, não uma qualidade fixa do
arquivo. Um catálogo institucional pode ser fonte secundária para estudar
as cartas que descreve, mas fonte primária para investigar práticas de
catalogação. Do mesmo modo, um artigo historiográfico é secundário para
estudar o período que interpreta, mas pode ser primário em uma pesquisa
sobre a historiografia daquele tema.

A classificação também não estabelece uma hierarquia automática de
verdade: fontes primárias e secundárias exigem crítica de autoria,
finalidade, contexto, mediação, ausências e limites. Uma tabela criada por
transcrição ou OCR é um **dado derivado**. Ela não substitui o documento e
deve permanecer ligada à fonte e às decisões que a produziram.

Fontes governamentais, institucionais e documentais exigem perguntas
distintas: quem produziu o registro, com qual finalidade, sob quais
categorias e com que condições de acesso?

**Referência:** American Historical Association (2023), seção “Shared
Values of Historians”. Dados completos em `referencias.md`.

Classificar a fonte esclarece seu uso, mas ainda não decide quais registros
entram no corpus. Para tornar essa passagem justificável, os critérios
precisam ser definidos antes da filtragem.

## 3. Critérios antes da filtragem

Um critério deve informar campo, regra, justificativa e tratamento dos
casos limítrofes. Para o experimento, adotaremos:

- período de 1890 a 1900;
- item localizado;
- representação digital disponível;
- acesso público ou mediante autorização.

Preveja quais grupos e instituições poderão perder presença.

In [ ]:
import pandas as pd

catalogo = pd.read_csv("dados/catalogo_fontes.csv")
criterio_periodo = catalogo["ano"].between(1890, 1900)
criterio_localizado = catalogo["localizado"].eq("sim")
criterio_digital = catalogo["digitalizado"].eq("sim")
criterio_acesso = catalogo["condicao_acesso"].isin(
    ["público", "mediante autorização"]
)

corpus = catalogo[
    criterio_periodo
    & criterio_localizado
    & criterio_digital
    & criterio_acesso
].copy()
corpus[["id_fonte", "instituicao", "ano", "grupo_representado"]]

A filtragem produz um corpus, mas também produz exclusões. A próxima etapa
registra esse outro resultado da seleção para que a decisão possa ser
auditada e revista.

## 4. Registrar exclusões

Reprodutibilidade não exige apenas uma lista final. Exige saber por que um
registro ficou de fora. O código abaixo produz um diagnóstico; a
justificativa substantiva continua sendo responsabilidade da pesquisa.

In [ ]:
motivos = pd.DataFrame(
    {
        "id_fonte": catalogo["id_fonte"],
        "fora_periodo": ~criterio_periodo,
        "nao_localizado": ~criterio_localizado,
        "sem_digitalizacao": ~criterio_digital,
        "acesso_incompativel": ~criterio_acesso,
    }
)
motivos["incluido"] = ~motivos.iloc[:, 1:].any(axis=1)
motivos

### Interpretação

A tabela de motivos descreve o efeito das regras. Agora é necessário
interpretá-la: uma contagem não explica, por si só, se a perda decorre do
fenômeno estudado, da infraestrutura ou de uma decisão da pesquisa.

1. Quantos registros foram excluídos por cada regra?
2. Um registro pode ter mais de um motivo?
3. “Não digitalizado” é uma propriedade do fenômeno ou da infraestrutura?
4. Seria possível consultar presencialmente parte do material excluído?

Escreva aqui.

Depois de interpretar o exemplo, transfira a mesma lógica para seu
projeto. A atividade converte conceitos e diagnóstico computacional em um
protocolo de seleção argumentado.

## Atividade — protocolo de seleção

Para seu projeto, registre:

**População de interesse:** Escreva aqui.

**População acessível e condições de acesso:** Escreva aqui.

**Unidade de análise:** Escreva aqui.

**Fontes e relação com a pergunta:** Escreva aqui.

**Critérios de inclusão, justificativas e evidências:** Escreva aqui.

**Critérios de exclusão e casos limítrofes:** Escreva aqui.

**Como as exclusões serão registradas:** Escreva aqui.

Ao terminar, verifique se outra pessoa conseguiria reconstruir sua decisão
sem adivinhar critérios. Essa verificação prepara a síntese e o exame de
cobertura do notebook seguinte.

## Síntese e leituras

Selecionar é construir o alcance da análise. Uma regra tecnicamente clara
pode continuar substantivamente inadequada. Rodrigues (2020) mostra como
a elaboração de uma base histórica envolve escolhas metodológicas e
éticas; Gebru et al. (2021) recomendam documentar motivação, composição,
coleta e usos. Dados completos: `referencias.md`.

Leve seu protocolo de seleção ao Notebook 02. O corpus escolhido será ali
comparado à cobertura desejada e à acessível, tornando visíveis perdas,
vieses e silêncios.